### Subset analyze

#### age

In [ ]:
import pandas as pd
df_final = pd.read_pickle("*******.pkl")
df_final

In [ ]:
df_final['label'].value_counts()

In [ ]:

selected_features = [
  'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       
       'age_at_admission', 'gender', 'race_group', 
       
       'aids', 'cerebrovascular_disease',
       #'chronic_pulmonary_disease', 
       'congestive_heart_failure', 'dementia',
       'diabetes_with_cc', 'diabetes_without_cc', 'malignant_cancer',
       'metastatic_solid_tumor', 'mild_liver_disease', 'myocardial_infarction',
       'paraplegia', 'peptic_ulcer_disease', 'peripheral_vascular_disease',
       'renal_disease', 'rheumatic_disease', 'severe_liver_disease'
      ]   

In [ ]:
df_final_above65 = df_final[df_final['age_at_admission'] > 65]
df_final_above65

In [ ]:
df_final_above65['subject_id'].nunique()

In [ ]:
df_final_above65['label'].value_counts()

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.svm import SVC

df_final = df_final_above65.copy()   

# Data preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

# Split the data by subject to create an independent test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Proportion of positive cases in the test set: {y_test.mean():.1%}")

# Build a pipeline using fixed optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

# Perform out-of-fold (OOF) analysis on the train+validation set to determine the optimal threshold

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

#Compute the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr


# Specify the minimum required sensitivity
min_sensitivity = 0.70         

# Filter for all thresholds that satisfy sensitivity ≥ `min_sensitivity`
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower the `min_sensitivity` value")
    optimal_threshold = 0.5
else:
    # Among the thresholds that meet the criteria, select the one with the highest specificity
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target minimum sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponding sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

#Train the final model on the complete train+validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Generate predictions on the independent test set using the optimal threshold

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)    


auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

In [ ]:
import pandas as pd
df_final = pd.read_pickle("*******.pkl")
df_final

In [ ]:
df_final['label'].value_counts()

In [ ]:

selected_features = [
  'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       
       'age_at_admission', 'gender', 'race_group', 
       
       'aids', 'cerebrovascular_disease',
       #'chronic_pulmonary_disease', 
       'congestive_heart_failure', 'dementia',
       'diabetes_with_cc', 'diabetes_without_cc', 'malignant_cancer',
       'metastatic_solid_tumor', 'mild_liver_disease', 'myocardial_infarction',
       'paraplegia', 'peptic_ulcer_disease', 'peripheral_vascular_disease',
       'renal_disease', 'rheumatic_disease', 'severe_liver_disease'
      ]   

In [ ]:
df_final_above65 = df_final[df_final['age_at_admission'] <= 65]
df_final_above65

In [ ]:
df_final_above65['subject_id'].nunique()

In [ ]:
df_final_above65['label'].value_counts()

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.svm import SVC

df_final = df_final_above65.copy()    

# Data preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

#Split the data by subject to create an independent test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Proportion of positive cases in the test set: {y_test.mean():.1%}")

# Build a pipeline using fixed optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

# Perform out-of-fold (OOF) analysis on the train+validation set to determine the optimal threshold

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# Compute the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr


# Specify the minimum required sensitivity
min_sensitivity = 0.70       

# Filter for all thresholds that satisfy sensitivity ≥ `min_sensitivity`
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower the `min_sensitivity` value")
    optimal_threshold = 0.5
else:
    # Among the thresholds that meet the criteria, select the one with the highest specificity
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target minimum sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponding sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

# Train the final model on the complete train+validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Generate predictions on the independent test set using the optimal threshold

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)    

auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

#### gender

In [ ]:
import pandas as pd

df_final = pd.read_pickle("*****.pkl")
df_final

In [ ]:
df_final_male = df_final[df_final['gender'] == 0]
df_final_male

In [ ]:
df_final_female = df_final[df_final['gender'] == 1]
df_final_female

In [ ]:
df_final_female['subject_id'].nunique()

In [ ]:
df_final_male['subject_id'].nunique()

In [ ]:

selected_features = [
'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       
       'age_at_admission', 'gender', 'race_group', 
       
       'aids', 'cerebrovascular_disease',
       #'chronic_pulmonary_disease', 
       'congestive_heart_failure', 'dementia',
       'diabetes_with_cc', 'diabetes_without_cc', 'malignant_cancer',
       'metastatic_solid_tumor', 'mild_liver_disease', 'myocardial_infarction',
       'paraplegia', 'peptic_ulcer_disease', 'peripheral_vascular_disease',
       'renal_disease', 'rheumatic_disease', 'severe_liver_disease'
      ]   

In [ ]:
df_final = df_final_male.copy()   

# Data preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

# Split the data by subject to create an independent test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Proportion of positive cases in the test set: {y_test.mean():.1%}")

# build a pipeline using fixed optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

#Perform out-of-fold (OOF) analysis on the train+validation set to determine the optimal threshold

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# compute the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr


# Specify the minimum required sensitivity
min_sensitivity = 0.70       

# Filter for all thresholds that satisfy sensitivity ≥ `min_sensitivity`
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower the `min_sensitivity` value")
    optimal_threshold = 0.5
else:
    # Among the thresholds that meet the criteria, select the one with the highest specificity
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target minimum sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponding sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

# Train the final model on the complete train+validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Generate predictions on the independent test set using the optimal threshold

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)   


auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

In [ ]:
df_final = df_final_female.copy()    

# Data preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

# Split the data by subject to create an independent test set

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Proportion of positive cases in the test set: {y_test.mean():.1%}")

# uild a pipeline using fixed optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

# Perform out-of-fold (OOF) analysis on the train+validation set to determine the optimal threshold

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# compute the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr

# Specify the minimum required sensitivity
min_sensitivity = 0.70     

# Filter for all thresholds that satisfy sensitivity ≥ `min_sensitivity`
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower the `min_sensitivity` value")
    optimal_threshold = 0.5
else:
    # Among the thresholds that meet the criteria, select the one with the highest specificity
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target minimum sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponding sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

# Train the final model on the complete train+validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Generate predictions on the independent test set using the optimal threshold
proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)    


auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

#### disease

In [ ]:
import pandas as pd

df_final = pd.read_pickle("******.pkl")
df_final

In [ ]:
df_final['subject_id'].nunique()

In [ ]:
df_final.columns

In [ ]:
# read pkl files
import pandas as pd

final_selected_patients_copd_asthma = pd.read_pickle("*******_copd_asthma10679.pkl")
final_selected_patients_copd_asthma

In [ ]:
final_selected_patients_copd_asthma['disease'].value_counts()

In [ ]:
final_selected_patients_copd_asthma['label'].value_counts()

In [ ]:
final_selected_patients_copd_asthma['subject_id'].nunique()

In [ ]:
# delete the subject_id that have two disease labels
ids_multiple_diseases = final_selected_patients_copd_asthma.groupby('subject_id')['disease'].nunique()
ids_multiple_diseases = ids_multiple_diseases[ids_multiple_diseases > 1].index.tolist()
final_selected_patients_copd_asthma = final_selected_patients_copd_asthma[
    ~final_selected_patients_copd_asthma['subject_id'].isin(ids_multiple_diseases)
]
final_selected_patients_copd_asthma

In [ ]:
final_selected_patients_copd_asthma

In [ ]:
# delete the subject_id that have two disease labels
ids_multiple_diseases = final_selected_patients_copd_asthma.groupby('subject_id')['disease'].nunique()
ids_multiple_diseases = ids_multiple_diseases[ids_multiple_diseases > 1].index.tolist()
ids_multiple_diseases

In [ ]:
# give the disease label from final_selected_patients_copd_asthma to df_final based on subject_id
df_final = df_final.merge(
    final_selected_patients_copd_asthma[['subject_id', 'disease']].drop_duplicates(),
    on='subject_id',
    how='left'
)

In [ ]:
df_final

In [ ]:
# assign disease labels asthma as 0, copd as 1 
df_final['label_disease'] = df_final['disease'].map({'asthma': 0, 'copd': 1})
df_final['label_disease'].value_counts()

In [ ]:
df_final_copd = df_final[df_final['label_disease'] == 1]
df_final_asthma = df_final[df_final['label_disease'] == 0]

In [ ]:
df_final_copd

In [ ]:
df_final_copd['subject_id'].nunique()

In [ ]:
df_final_copd['label'].value_counts()

In [ ]:
df_final_asthma['label'].value_counts()

In [ ]:
df_final_asthma['subject_id'].nunique()

In [ ]:

selected_features = [
'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       
       'age_at_admission', 'gender', 'race_group', 
       
       'aids', 'cerebrovascular_disease',
       #'chronic_pulmonary_disease', 
       'congestive_heart_failure', 'dementia',
       'diabetes_with_cc', 'diabetes_without_cc', 'malignant_cancer',
       'metastatic_solid_tumor', 'mild_liver_disease', 'myocardial_infarction',
       'paraplegia', 'peptic_ulcer_disease', 'peripheral_vascular_disease',
       'renal_disease', 'rheumatic_disease', 'severe_liver_disease'
      ]   

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.svm import SVC

df_final = df_final_asthma.copy()    

# Data preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

# Split the data by subject to create an independent test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Proportion of positive cases in the test set: {y_test.mean():.1%}")

# build a pipeline using fixed optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

# Perform out-of-fold (OOF) analysis on the train+validation set to determine the optimal threshold
outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# compute the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr

# Specify the minimum required sensitivity
min_sensitivity = 0.70       

# Filter for all thresholds that satisfy sensitivity ≥ `min_sensitivity`
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower the `min_sensitivity` value")
    optimal_threshold = 0.5
else:
    # Among the thresholds that meet the criteria, select the one with the highest specificity
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target minimum sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponding sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

# Train the final model on the complete train+validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Generate predictions on the independent test set using the optimal threshold

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)   


auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

In [ ]:
df_final = df_final_copd.copy()   

# Data preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

# Split the data by subject to create an independent test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Proportion of positive cases in the test set: {y_test.mean():.1%}")

# build a pipeline using fixed optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

# Perform out-of-fold (OOF) analysis on the train+validation set to determine the optimal threshold

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# compute the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr

# Specify the minimum required sensitivity
min_sensitivity = 0.70         

# Filter for all thresholds that satisfy sensitivity ≥ `min_sensitivity`
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower the `min_sensitivity` value")
    optimal_threshold = 0.5
else:
    # Among the thresholds that meet the criteria, select the one with the highest specificity
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target minimum sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponding sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

# Train the final model on the complete train+validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Generate predictions on the independent test set using the optimal threshold

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)   


auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")